In [ ]:
import subprocess
import sys
import time
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

def run_cmd(*args, timeout=300, check=True):
    print("$", " ".join(str(a) for a in args), flush=True)
    result = subprocess.run(
        [str(a) for a in args],
        text=True,
        timeout=timeout,
        check=check
    )
    print("Hoàn tất.", flush=True)
    return result

# 1. Cập nhật package
run_cmd(
    "apt-get",
    "update",
    "-o", "Acquire::Retries=2",
    "-o", "Acquire::http::Timeout=30",
    "-o", "Acquire::https::Timeout=30",
    timeout=180
)

# 2. Cài PostgreSQL + curl
run_cmd(
    "apt-get",
    "install",
    "-y",
    "--no-install-recommends",
    "postgresql",
    "postgresql-contrib",
    "curl",
    timeout=300
)

# 3. Cài Java 21 + Maven + Locust
run_cmd(
    "apt-get",
    "install",
    "-y",
    "--no-install-recommends",
    "openjdk-21-jdk",
    "maven",
    timeout=300
)

run_cmd(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "locust==2.32.9",
    "requests",
    timeout=300
)

# 4. Khởi động PostgreSQL
run_cmd(
    "service",
    "postgresql",
    "start",
    timeout=60
)

print("Cell 1 hoàn tất.")

In [ ]:
Bước 2: Thiết lập Database PostgreSQL (Tạo DB ktpm_chat & cấp quyền)

In [4]:
import subprocess

def psql(sql, database="postgres"):
    result = subprocess.run(
        ["runuser", "-u", "postgres", "--", "psql", "-d", database, "-tAc", sql],
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout.strip()

# Đặt password cho postgres
psql("ALTER USER postgres WITH PASSWORD '123';")

# Tạo DB nếu chưa có
if psql("SELECT 1 FROM pg_database WHERE datname='ktpm_chat'") != "1":
    psql("CREATE DATABASE ktpm_chat OWNER postgres;")
    print("Đã tạo database ktpm_chat.")
else:
    print("Database ktpm_chat đã tồn tại.")

# Kiểm tra nhanh
print(psql("SELECT current_database(), current_user", database="ktpm_chat"))

print("Cell 2 hoàn tất.")

Database ktpm_chat đã tồn tại.
ktpm_chat|postgres
Cell 2 hoàn tất.


In [28]:
import os
import subprocess
import time
import requests
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "ktpm-chatroom"
BASE_URL = "http://127.0.0.1:8080"
LOG_FILE = WORK / "backend.log"

def run_cmd(*args, timeout=600, check=True, cwd=None):
    print("$", " ".join(str(a) for a in args), flush=True)
    result = subprocess.run(
        [str(a) for a in args],
        text=True,
        timeout=timeout,
        check=check,
        cwd=str(cwd) if cwd else None
    )
    return result

# Dọn repo cũ
if REPO_DIR.exists():
    run_cmd("rm", "-rf", str(REPO_DIR), timeout=60)

# Clone repo
run_cmd(
    "git",
    "clone",
    "--depth",
    "1",
    "https://github.com/KhanhVNMC/ktpm-chatroom.git",
    str(REPO_DIR),
    timeout=300
)

# Build ứng dụng Java
run_cmd(
    "mvn",
    "-q",
    "clean",
    "package",
    "dependency:copy-dependencies",
    "-DincludeScope=runtime",
    "-DskipTests",
    cwd=REPO_DIR,
    timeout=600
)

jar_file = REPO_DIR / "target" / "ktpm-chat-1.0.0-SNAPSHOT.jar"
lib_dir = REPO_DIR / "target" / "dependency"

if not jar_file.exists():
    raise FileNotFoundError(f"Không tìm thấy JAR: {jar_file}")

if not lib_dir.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục dependency: {lib_dir}")

# Dừng tiến trình backend cũ nếu có
subprocess.run(["pkill", "-f", "dev.ktpm.App"], check=False)

# Môi trường kết nối DB
env = os.environ.copy()
env.update({
    "DB_HOST": "127.0.0.1",
    "DB_PORT": "5432",
    "DB_NAME": "ktpm_chat",
    "DB_USER": "postgres",
    "DB_PASSWORD": "123"
})

# Mở file log
with open(LOG_FILE, "w", encoding="utf-8") as log_handle:
    print("Khởi động Java backend...", flush=True)

    process = subprocess.Popen(
        [
            "java",
            "-cp",
            f"{jar_file}:{lib_dir}/*",
            "dev.ktpm.App"
        ],
        cwd=str(REPO_DIR),
        env=env,
        stdout=log_handle,
        stderr=subprocess.STDOUT
    )

    print("Java backend PID:", process.pid, flush=True)

    healthy = False
    for attempt in range(35):
        time.sleep(2)
        try:
            res = requests.get(f"{BASE_URL}/api/health", timeout=3)
            print(f"Lần {attempt + 1}: HTTP {res.status_code} - {res.text}", flush=True)
            if res.status_code == 200 and res.text.strip() == "OK":
                healthy = True
                break
        except Exception as e:
            print(f"Lần {attempt + 1}: backend chưa sẵn sàng - {e}", flush=True)

        if process.poll() is not None:
            print("Backend đã dừng bất thường.", flush=True)
            break

    if not healthy:
        print("\nLog backend:\n", flush=True)
        if LOG_FILE.exists():
            content = LOG_FILE.read_text(encoding="utf-8")
            print(content[-4000:], flush=True)
        raise RuntimeError("Java backend không khởi động được.")

print("Cell 3 hoàn tất. Backend đang chạy ở:", BASE_URL)

$ rm -rf /kaggle/working/ktpm-chatroom


Exception ignored in: <bound method _ForkHooks.after_fork_in_child of <gevent.threading._ForkHooks object at 0x7c628b5486e0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 447, in after_fork_in_child
    self._stop_running_greenlets_in_child()
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 435, in _stop_running_greenlets_in_child
    handle._set_done(enter_hub=False)
TypeError: _ThreadHandle._set_done() takes no keyword arguments


$ git clone --depth 1 https://github.com/KhanhVNMC/ktpm-chatroom.git /kaggle/working/ktpm-chatroom


Exception ignored in: <bound method _ForkHooks.after_fork_in_child of <gevent.threading._ForkHooks object at 0x7c628b5486e0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 447, in after_fork_in_child
    self._stop_running_greenlets_in_child()
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 435, in _stop_running_greenlets_in_child
    handle._set_done(enter_hub=False)
TypeError: _ThreadHandle._set_done() takes no keyword arguments
Cloning into '/kaggle/working/ktpm-chatroom'...


$ mvn -q clean package dependency:copy-dependencies -DincludeScope=runtime -DskipTests


Exception ignored in: <bound method _ForkHooks.after_fork_in_child of <gevent.threading._ForkHooks object at 0x7c628b5486e0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 447, in after_fork_in_child
    self._stop_running_greenlets_in_child()
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 435, in _stop_running_greenlets_in_child
    handle._set_done(enter_hub=False)
TypeError: _ThreadHandle._set_done() takes no keyword arguments
Exception ignored in: <bound method _ForkHooks.after_fork_in_child of <gevent.threading._ForkHooks object at 0x7c628b5486e0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 447, in after_fork_in_child
    self._stop_running_greenlets_in_child()
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 435, in _stop_running_greenlets_in_child
    handle._set_done(enter_hub=False)
TypeError: _

Khởi động Java backend...


Exception ignored in: <bound method _ForkHooks.after_fork_in_child of <gevent.threading._ForkHooks object at 0x7c628b5486e0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 447, in after_fork_in_child
    self._stop_running_greenlets_in_child()
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 435, in _stop_running_greenlets_in_child
    handle._set_done(enter_hub=False)
TypeError: _ThreadHandle._set_done() takes no keyword arguments


Java backend PID: 6811
Lần 1: backend chưa sẵn sàng - HTTPConnectionPool(host='127.0.0.1', port=8080): Max retries exceeded with url: /api/health (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8080): Failed to establish a new connection: [Errno 111] Connection refused"))
Lần 2: backend chưa sẵn sàng - HTTPConnectionPool(host='127.0.0.1', port=8080): Max retries exceeded with url: /api/health (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8080): Failed to establish a new connection: [Errno 111] Connection refused"))
Lần 3: HTTP 200 - OK
Cell 3 hoàn tất. Backend đang chạy ở: http://127.0.0.1:8080


In [17]:
import requests

BASE_URL = "http://127.0.0.1:8080"

def api(method, path, payload=None, session_id=None):
    headers = {}
    if session_id:
        headers["Authorization"] = f"Bearer {session_id}"
    r = requests.request(
        method,
        BASE_URL + path,
        json=payload,
        headers=headers,
        timeout=10
    )
    print(f"{method.upper()} {path} -> {r.status_code}")
    print(r.text[:500])
    return r

# 1) Register user
username = "kaggle_user_1"
password = "123456"
display_name = "Kaggle User 1"

# 2) Login để lấy session_id
r = api("POST", "/api/auth/login", {
    "username": username,
    "password": password
})
if r.status_code != 200:
    raise RuntimeError(f"Login failed: {r.text}")

session_id = r.json()["session_id"]
print("SESSION_ID =", session_id)

# 3) Tạo room
r = api("POST", "/api/chatrooms", {
    "name": "Room Kaggle Test",
    "motd": "Hello from Kaggle",
    "is_public": True
}, session_id=session_id)

if r.status_code != 201:
    raise RuntimeError(f"Create room failed: {r.text}")

room = r.json()
room_uuid = room["room_uuid"]
invite_code = room["invite_code"]

print("ROOM_UUID =", room_uuid)
print("INVITE_CODE =", invite_code)

print("Cell 4 hoàn tất.")

POST /api/auth/login -> 200
{"session_id":"d1c52b4b-24f9-4fb0-8390-989088f59319"}
SESSION_ID = d1c52b4b-24f9-4fb0-8390-989088f59319
POST /api/chatrooms -> 201
{"room_uuid":"cf1cb5f6-8fad-4497-b758-61d557060c24","invite_code":null,"invite_link":null}
ROOM_UUID = cf1cb5f6-8fad-4497-b758-61d557060c24
INVITE_CODE = None
Cell 4 hoàn tất.


In [18]:
import time
import requests

BASE_URL = "http://127.0.0.1:8080"

def api(method, path, payload=None, session_id=None):
    headers = {}
    if session_id:
        headers["Authorization"] = f"Bearer {session_id}"
    r = requests.request(
        method,
        BASE_URL + path,
        json=payload,
        headers=headers,
        timeout=10
    )
    print(f"{method.upper()} {path} -> {r.status_code}", flush=True)
    try:
        body = r.json()
        if isinstance(body, dict):
            print(body, flush=True)
        else:
            print(body, flush=True)
    except Exception:
        print(r.text[:500], flush=True)
    return r

# 1) health check
r = api("GET", "/api/health")
assert r.status_code == 200 and r.text.strip() == "OK", r.text

# 2) register
username = f"kaggle_smoke_{int(time.time() * 1000)}"
password = "123456"
display_name = "Kaggle Smoke"
r = api("POST", "/api/auth/register", {
    "username": username,
    "password": password,
    "display_name": display_name
})
assert r.status_code in (200, 201), r.text

# 3) login
r = api("POST", "/api/auth/login", {
    "username": username,
    "password": password
})
assert r.status_code == 200, r.text
session_id = r.json()["session_id"]

# 4) me
r = api("GET", "/api/auth/me", session_id=session_id)
assert r.status_code == 200, r.text

# 5) create room
r = api("POST", "/api/chatrooms", {
    "name": "Smoke Room",
    "motd": "Kaggle smoke test",
    "is_public": True
}, session_id=session_id)
assert r.status_code == 201, r.text
room = r.json()
room_uuid = room["room_uuid"]

# 6) list public rooms
r = api("GET", "/api/chatrooms/listing?limit=10", session_id=session_id)
assert r.status_code == 200, r.text

# 7) room overview
r = api("GET", f"/api/chatrooms/{room_uuid}/overview", session_id=session_id)
assert r.status_code == 200, r.text

# 8) my rooms
r = api("GET", "/api/chatrooms/mine?limit=10", session_id=session_id)
assert r.status_code == 200, r.text

# 9) logout
r = api("POST", "/api/auth/logout", session_id=session_id)
assert r.status_code == 200, r.text

print("Cell 5 hoàn tất: smoke test API thành công.")

GET /api/health -> 200
OK
POST /api/auth/register -> 201
{'uuid': 'f8924c6d-56bd-453b-b4bd-d6a056a6d5c4', 'username': 'kaggle_smoke_1793198449577', 'display_name': 'Kaggle Smoke', 'avatar': None, 'presence': 'OFFLINE', 'status_text': None}
POST /api/auth/login -> 200
{'session_id': 'fafcaf9e-5cdd-4add-b864-1a7da4fe8911'}
GET /api/auth/me -> 200
{'uuid': 'f8924c6d-56bd-453b-b4bd-d6a056a6d5c4', 'username': 'kaggle_smoke_1793198449577', 'display_name': 'Kaggle Smoke', 'avatar': None, 'presence': 'ONLINE', 'status_text': None}
POST /api/chatrooms -> 201
{'room_uuid': 'd07685ae-9991-4595-bdf6-10b1b67e1c75', 'invite_code': None, 'invite_link': None}
GET /api/chatrooms/listing?limit=10 -> 200
{'public_rooms': [{'roomUUID': 'd07685ae-9991-4595-bdf6-10b1b67e1c75', 'name': 'Smoke Room', 'motd': 'Kaggle smoke test', 'avatar': None, 'createdAt': 1793198449860, 'members': 1}, {'roomUUID': 'cf1cb5f6-8fad-4497-b758-61d557060c24', 'name': 'Room Kaggle Test', 'motd': 'Hello from Kaggle', 'avatar': None

In [19]:
from pathlib import Path

WORK = Path("/kaggle/working")
locust_text = '''import os
import uuid
from locust import HttpUser, task, between

BASE_URL = os.getenv("BASE_URL", "http://127.0.0.1:8080")

class ChatUser(HttpUser):
    host = BASE_URL
    wait_time = between(0.5, 2.0)

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.username = f"locust_{uuid.uuid4().hex[:10]}"
        self.password = "123456"
        self.session_id = None
        self.room_uuid = None

    def auth_header(self):
        return {"Authorization": f"Bearer {self.session_id}"} if self.session_id else {}

    def on_start(self):
        reg = self.client.post("/api/auth/register", json={
            "username": self.username,
            "password": self.password,
            "display_name": self.username
        }, name="auth/register")
        if reg.status_code not in (200, 201):
            if reg.status_code == 400:
                pass
            else:
                reg.raise_for_status()

        login = self.client.post("/api/auth/login", json={
            "username": self.username,
            "password": self.password
        }, name="auth/login")
        if login.status_code != 200:
            login.raise_for_status()
        self.session_id = login.json()["session_id"]

        room = self.client.post("/api/chatrooms", json={
            "name": f"Room {self.username}",
            "motd": "Locust test room",
            "is_public": True
        }, headers=self.auth_header(), name="chatrooms/create")
        if room.status_code == 201:
            self.room_uuid = room.json()["room_uuid"]

    @task(1)
    def health(self):
        self.client.get("/api/health", name="health")

    @task(3)
    def list_public_rooms(self):
        self.client.get("/api/chatrooms/listing?limit=10", name="chatrooms/listing")

    @task(3)
    def my_rooms(self):
        if self.session_id:
            self.client.get("/api/chatrooms/mine?limit=10", headers=self.auth_header(), name="chatrooms/mine")

    @task(4)
    def overview(self):
        if self.room_uuid:
            self.client.get(f"/api/chatrooms/{self.room_uuid}/overview", headers=self.auth_header(), name="chatrooms/overview")
        else:
            self.client.get("/api/chatrooms/listing?limit=10", name="chatrooms/listing_fallback")

    @task(2)
    def me(self):
        if self.session_id:
            self.client.get("/api/auth/me", headers=self.auth_header(), name="auth/me")
'''

(WORK / "locustfile.py").write_text(locust_text, encoding="utf-8")
print("Cell 6 hoàn tất: locustfile.py đã được tạo.")

Cell 6 hoàn tất: locustfile.py đã được tạo.


In [30]:
import subprocess
from pathlib import Path
import time
import requests

WORK = Path("/kaggle/working")
REPORT_DIR = WORK / "locust_reports"
REPORT_DIR.mkdir(exist_ok=True)

# health check cho backend
for i in range(20):
    try:
        r = requests.get("http://127.0.0.1:8080/api/health", timeout=3)
        if r.status_code == 200 and r.text.strip() == "OK":
            print("Backend ready")
            break
    except Exception as e:
        print(f"backend chưa ready: {e}")
    time.sleep(2)

cmd = [
    "locust",
    "-f", str(WORK / "locustfile.py"),
    "--headless",
    "-u", "100",
    "-r", "5",
    "-t", "60s",
    "--host", "http://127.0.0.1:8080",
    "--csv", str(REPORT_DIR / "locust_result"),
    "--html", str(REPORT_DIR / "locust_report.html")
]

print("Bắt đầu load test 100 user trong 60s...", flush=True)
subprocess.run(cmd, check=True)
print("Cell 7 hoàn tất.")

Backend ready
Bắt đầu load test 100 user trong 60s...


Exception ignored in: <bound method _ForkHooks.after_fork_in_child of <gevent.threading._ForkHooks object at 0x7c628b5486e0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 447, in after_fork_in_child
    self._stop_running_greenlets_in_child()
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 435, in _stop_running_greenlets_in_child
    handle._set_done(enter_hub=False)
TypeError: _ThreadHandle._set_done() takes no keyword arguments
[2026-10-28 15:51:49,530] 739ec6536580/INFO/locust.main: Starting Locust 2.32.9
[2026-10-28 15:51:49,532] 739ec6536580/INFO/locust.main: Run time limit set to 60 seconds
Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|---

Cell 7 hoàn tất.


[2026-10-28 15:52:49,397] 739ec6536580/INFO/locust.main: Shutting down (exit code 0)
Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
POST     auth/login                                                                       100     0(0.00%) |    192     141     288    190 |    1.68        0.00
GET      auth/me                                                                          639     0(0.00%) |      4       2      27      4 |   10.71        0.00
POST     auth/register                                                                    100     0(0.00%) |    211     146     322    210 |    1.68        0.00
POST     chatrooms/create                                                                 100     0(0.00%) |  

In [ ]:
## Bước 8: Trích xuất kết quả thống kê và tạo báo cáo HTML

In [31]:
import json
from pathlib import Path
from statistics import mean

WORK = Path("/kaggle/working")
REPORT_DIR = WORK / "locust_reports"

# đọc CSV summary
# locust_result_stats.csv format:
# Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,Failures/s,50%,66%,75%,80%,90%,95%,98%,99%,99.9%
# nếu file chưa có, bỏ qua

csv_path = REPORT_DIR / "locust_result_stats.csv"
html_path = REPORT_DIR / "locust_report.html"

summary = {}
if csv_path.exists():
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    if len(lines) > 2:
        # đọc dòng cuối cùng chứa tổng hợp
        last = lines[-1].strip()
        if last:
            cols = last.split(",")
            if len(cols) >= 10:
                summary = {
                    "request_count": cols[2] if len(cols) > 2 else "",
                    "failure_count": cols[3] if len(cols) > 3 else "",
                    "avg_response_ms": cols[5] if len(cols) > 5 else "",
                    "min_response_ms": cols[6] if len(cols) > 6 else "",
                    "max_response_ms": cols[7] if len(cols) > 7 else "",
                    "req_per_sec": cols[9] if len(cols) > 9 else "",
                    "fail_per_sec": cols[10] if len(cols) > 10 else "",
                }

print("Summary:", summary, flush=True)

# HTML báo cáo đơn giản
html = f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>Locust Report</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; background: #f7f7f7; }}
    .card {{ background: white; padding: 20px; border-radius: 10px; box-shadow: 0 1px 3px rgba(0,0,0,0.08); margin-bottom: 16px; }}
    h1 {{ margin-top: 0; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
    th {{ background: #f0f0f0; }}
    .ok {{ color: green; font-weight: bold; }}
    .warn {{ color: orange; font-weight: bold; }}
  </style>
</head>
<body>
  <div class="card">
    <h1>Locust Load Test Report</h1>
    <p><strong>Thời gian:</strong> {__import__('datetime').datetime.utcnow().isoformat()}Z</p>
    <p><strong>Backend:</strong> http://127.0.0.1:8080</p>
  </div>

  <div class="card">
    <h2>Kết quả tổng hợp</h2>
    <table>
      <tr><th>Mục</th><th>Giá trị</th></tr>
      <tr><td>Request Count</td><td>{summary.get('request_count', 'N/A')}</td></tr>
      <tr><td>Failure Count</td><td>{summary.get('failure_count', 'N/A')}</td></tr>
      <tr><td>Avg Response Time (ms)</td><td>{summary.get('avg_response_ms', 'N/A')}</td></tr>
      <tr><td>Min Response Time (ms)</td><td>{summary.get('min_response_ms', 'N/A')}</td></tr>
      <tr><td>Max Response Time (ms)</td><td>{summary.get('max_response_ms', 'N/A')}</td></tr>
      <tr><td>Requests/s</td><td>{summary.get('req_per_sec', 'N/A')}</td></tr>
      <tr><td>Failures/s</td><td>{summary.get('fail_per_sec', 'N/A')}</td></tr>
    </table>
  </div>

  <div class="card">
    <h2>Thông tin thêm</h2>
    <p>Report HTML gốc: <a href=\"locust_report.html\">locust_report.html</a></p>
    <p>CSV thống kê: <a href=\"locust_result_stats.csv\">locust_result_stats.csv</a></p>
    <p class=\"ok\">Nếu Failure Count = 0 thì test API đang chạy ổn.</p>
  </div>
</body>
</html>
"""

REPORT_DIR.mkdir(exist_ok=True, parents=True)
( REPORT_DIR / "summary_report.html" ).write_text(html, encoding="utf-8")

print("Cell 8 hoàn tất. Báo cáo HTML:", REPORT_DIR / "summary_report.html")

Summary: {'request_count': '4207', 'failure_count': '0', 'avg_response_ms': '17.90993864345037', 'min_response_ms': '1.690332999714883', 'max_response_ms': '322.03191299959144', 'req_per_sec': '71.15101847267768', 'fail_per_sec': '0.0'}
Cell 8 hoàn tất. Báo cáo HTML: /kaggle/working/locust_reports/summary_report.html


/tmp/ipykernel_49/108827024.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  <p><strong>Thời gian:</strong> {__import__('datetime').datetime.utcnow().isoformat()}Z</p>
